# Session 5 — Capstone: a Transformer block from scratch

Companion to [../numpy_pytorch_schedule.md](../numpy_pytorch_schedule.md). Everything from Sessions 1–4, assembled into the object this repo is about. If you can build and train this, you're fluent. Run it top to bottom; **print shapes at every step**.

Convention: `B` batch, `S` sequence, `D` model, `H` heads, `D_h = D/H` (matching [../NOTATION.md](../NOTATION.md), [../ARCHITECTURE.md](../ARCHITECTURE.md)).

In [4]:
import torch, torch.nn as nn, torch.nn.functional as F

## Step 1 — Scaled dot-product attention, checked against PyTorch

Write it from raw ops, then confirm it matches `F.scaled_dot_product_attention`. Matching means you got the scale, the transpose, and the softmax axis right — the real test.

In [5]:
# SDPA = Scaled Dot-Product Attention:  softmax(Q·Kᵀ / √d_k) · V   (Vaswani et al., 2017)
#   "scaled" = the / √d_k ;  "dot-product" = Q·Kᵀ ;  then softmax over keys, weighted sum of V.
def sdpa_manual(q, k, v):                     # q,k,v: (B,H,S,D_h)
    d_h = q.shape[-1]
    scores = (q @ k.transpose(-2, -1)) / d_h**0.5   # dot-product Q·Kᵀ, scaled by √d_h -> (B,H,S,S)
    A = F.softmax(scores, dim=-1)                    # softmax over the KEY axis (each query's weights sum to 1)
    return A @ v                                     # weighted average of the value vectors -> (B,H,S,D_h)

q, k, v = (torch.randn(2, 4, 6, 8) for _ in range(3))
mine = sdpa_manual(q, k, v)
ref  = F.scaled_dot_product_attention(q, k, v)       # PyTorch's fused SDPA (same formula, optimized)
print("shapes:", mine.shape, ref.shape)
print("matches PyTorch:", torch.allclose(mine, ref, atol=1e-5))   # True -> our math is right

shapes: torch.Size([2, 4, 6, 8]) torch.Size([2, 4, 6, 8])
matches PyTorch: True


## Step 2 — Multi-head attention

Project to `Q/K/V`, reshape `(B,S,D)→(B,H,S,D_h)`, attend per head, concat, project with `W_O`. The `transpose → contiguous → view` wrinkle from Session 3 shows up here.

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, D, H, causal=True):
        super().__init__()
        self.H, self.D_h, self.causal = H, D // H, causal   # H heads, each of width D_h = D/H
        self.qkv  = nn.Linear(D, 3 * D, bias=False)  # ONE wide matmul makes Q, K, V for all heads (see note below)
        self.proj = nn.Linear(D, D, bias=False)      # W_O: mixes the heads and writes back into the stream

    def forward(self, x):                            # x: (B, S, D)  — one D-vector per token
        B, S, D = x.shape
        # 1) project, then split the 3*D output into three (B, S, D) chunks: Q, K, V
        q, k, v = self.qkv(x).split(D, dim=2)
        # 2) split each (B, S, D) into H heads, then move the head axis up front:
        #    .view(B, S, H, D_h) reinterprets the D-wide vector at each token as H chunks of
        #      width D_h. No copy: D == H*D_h, so it just regroups the contiguous features
        #      (head h "owns" features [h*D_h : (h+1)*D_h]).   (B, S, D) -> (B, S, H, D_h)
        #    .transpose(1, 2) swaps the S and H axes ->        (B, S, H, D_h) -> (B, H, S, D_h)
        #      now (B, H) are leading "batch" axes, so attention runs independently per head.
        #      NOTE: transpose is a NO-COPY metadata swap (it just swaps two strides), so the
        #      result is NON-contiguous. That's fine here: scaled_dot_product_attention reads
        #      strided tensors directly, so no .contiguous() is needed at this point.
        q, k, v = (t.view(B, S, self.H, self.D_h).transpose(1, 2) for t in (q, k, v))
        # 3) scaled dot-product attention per head; is_causal=True adds the autoregressive
        #    mask (each position attends only to itself and earlier) for free
        out = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal)   # (B, H, S, D_h)
        # 4) concat the heads back to width D: (B, H, S, D_h) -> (B, S, H, D_h) -> (B, S, D)
        #    transpose(1,2) again makes it NON-contiguous (no copy, just swapped strides). Here we
        #    DO need .contiguous() because .view() requires contiguous memory (unlike SDPA above,
        #    which tolerates strides): .contiguous() allocates a fresh row-major buffer, then
        #    .view() merges the H and D_h axes back into one D-wide vector per token.
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        # 5) W_O mixes information across heads and produces the (B, S, D) result that gets
        #    added into the residual stream by the enclosing block
        return self.proj(out)

mha = MultiHeadAttention(D=64, H=4)
print("MHA out:", mha(torch.randn(2, 10, 64)).shape)   # (2, 10, 64) — same shape in and out

MHA out: torch.Size([2, 10, 64])


### Why one `nn.Linear(D, 3*D)` instead of three `nn.Linear(D, D)`?

A linear layer is just a matmul by a weight matrix, and three separate projections `W_Q, W_K, W_V` (each `D×D`) are **identical** to one `D×(3D)` matrix whose columns are those three concatenated: `W_qkv = [W_Q | W_K | W_V]`. Because matmul distributes over concatenated columns,

```
x @ [W_Q | W_K | W_V]  ==  [ x@W_Q | x@W_K | x@W_V ]
```

so the wide output already *contains* Q, K, V side by side — `.split(D, dim=2)` just carves it back into three `(B, S, D)` chunks. **Same math** as three separate layers; done as **one wider matmul purely for speed** (a single large matmul beats three small ones on a GPU — better utilization, fewer kernel launches). Proof:

In [8]:
D = 8
x = torch.randn(2, 5, D)                              # (B, S, D)
Wq, Wk, Wv = (torch.randn(D, D) for _ in range(3))   # three separate projections

combined = x @ torch.cat([Wq, Wk, Wv], dim=1)        # one wide matmul, W_qkv = [Wq|Wk|Wv] -> (B, S, 3D)
q, k, v = combined.split(D, dim=2)                    # carve back into three (B, S, D)
print("q == x@Wq:", torch.allclose(q, x @ Wq))        # True
print("k == x@Wk:", torch.allclose(k, x @ Wk))        # True
print("v == x@Wv:", torch.allclose(v, x @ Wv))        # True

q == x@Wq: True
k == x@Wk: True
v == x@Wv: True


Note `.contiguous()` before `.view()` — `transpose` made memory non-contiguous. `is_causal=True` gives the autoregressive mask for free.

## Step 3 — A pre-norm Transformer block

`x = x + attn(norm(x))` then `x = x + mlp(norm(x))` — residual + pre-norm from Part 3, with a GELU MLP at `4D` hidden width.

In [10]:
class Block(nn.Module):
    def __init__(self, D, H):
        super().__init__()
        # LayerNorm is an nn *layer* (not an F.* function) because it has learnable params:
        # a per-feature gain γ (=.weight) and shift β (=.bias), each shape (D,).
        self.ln1 = nn.LayerNorm(D); self.attn = MultiHeadAttention(D, H, causal=True)
        self.ln2 = nn.LayerNorm(D)
        # position-wise FFN: expand D -> 4D, GELU nonlinearity, project 4D -> D (acts per token)
        self.mlp = nn.Sequential(nn.Linear(D, 4*D, bias=False), nn.GELU(),
                                 nn.Linear(4*D, D, bias=False))

    def forward(self, x):                     # x: (B, S, D) — the residual stream
        # pre-norm residual: normalize a COPY, run the sublayer, ADD the result back to the
        # untouched stream (x + ...). The stream on the "x +" path is never normalized in place.
        x = x + self.attn(self.ln1(x))        # attention sublayer: mixes info ACROSS positions
        x = x + self.mlp(self.ln2(x))         # FFN sublayer:       transforms EACH token on its own
        return x

print("Block out:", Block(64, 4)(torch.randn(2, 10, 64)).shape)   # (2, 10, 64) — same shape in and out

Block out: torch.Size([2, 10, 64])


## Step 4 — A tiny GPT, and train it

Embed tokens + positions, stack blocks, final norm, tied LM head. Train on a repeating char corpus with the exact Session 4 loop.

In [11]:
class TinyGPT(nn.Module):
    def __init__(self, vocab, D=128, H=4, n_layer=3, block_size=64):
        super().__init__()
        self.block_size = block_size                  # max context length (positions we have embeddings for)
        self.tok = nn.Embedding(vocab, D)             # token id -> D-vector (lookup table, shape (vocab, D))
        self.pos = nn.Embedding(block_size, D)        # position index -> D-vector (learned absolute positions)
        # the stack of L Transformer blocks. nn.ModuleList (not a plain list!) so the blocks'
        # parameters are REGISTERED and show up in .parameters() (the Session-4 gotcha).
        self.blocks = nn.ModuleList(Block(D, H) for _ in range(n_layer))
        self.ln_f = nn.LayerNorm(D)                   # final norm: pre-norm never normalizes the stream in
                                                      # place, so tame its scale once before the head
        self.head = nn.Linear(D, vocab, bias=False)   # unembedding: D-vector -> vocab logits
        self.head.weight = self.tok.weight            # WEIGHT TYING: one (vocab, D) matrix serves BOTH the
                                                      # input embedding and the output projection — saves
                                                      # V*D params and ties input/output token reps
        self.apply(self._init)                        # recursively run _init on every submodule

    def _init(self, m):                               # called on each submodule by self.apply(...)
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)       # GPT-2 init: N(0, 0.02)  (see 2.3/02)

    def forward(self, idx, targets=None):             # idx: (B, S) token ids
        B, S = idx.shape
        # embed tokens AND positions, then add: (B,S,D) + (S,D) broadcasts over the batch.
        # the position embedding is what gives the model a sense of order (attention alone is order-blind).
        x = self.tok(idx) + self.pos(torch.arange(S, device=idx.device))
        for b in self.blocks:                         # refine the residual stream through the L blocks
            x = b(x)
        logits = self.head(self.ln_f(x))              # final norm, then project to vocab -> (B, S, vocab)
        if targets is None:
            return logits                             # inference / generation: just return the logits
        # training: flatten (B,S,vocab) -> (B*S, vocab) and (B,S) -> (B*S,), because F.cross_entropy
        # wants (N, C) logits + (N,) integer targets. targets are the NEXT token at each position.
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

**`self.apply(self._init)` — what it does.** `apply(fn)` is a **built-in `nn.Module` method**: it walks the whole module tree and calls `fn(module)` on **every submodule and on `self`** (children first, self last). It's just a clean shorthand for `for m in self.modules(): fn(m)` — the standard idiom for **weight initialization** (nanoGPT does exactly this).

`_init` is **not special to PyTorch** — it's an ordinary method you defined; the name is arbitrary (`init_weights`, `foo`, anything works). The leading underscore is only a Python convention for "internal helper." `apply` just needs a function that takes a module; that function `isinstance`-checks each one and sets its `.weight`/`.bias`.

You only do this to **override** defaults: `nn.Linear`/`nn.Embedding` already initialize sensibly, but here we want GPT-2's `N(0, 0.02)` (Part [2.3/02](../part2_neural_network_fundamentals/2.3_init_normalization/02_xavier_kaiming_modern.md)). Modules the function doesn't match keep their defaults. (Aside: HuggingFace's `PreTrainedModel` *does* look for a method literally named `_init_weights`, but that's a library convention on top of core PyTorch, not a PyTorch rule.)

### The loss line, in detail: `F.cross_entropy(logits.view(-1, V), targets.view(-1))`

**What `view(-1, ...)` does — `-1` means "infer this dimension."** You give the sizes you care about and put `-1` where PyTorch should work out the rest from the total number of elements. `logits` is `(B, S, vocab)`; `logits.view(-1, logits.size(-1))` says "reshape to `(?, vocab)`," so PyTorch infers `? = B*S` → shape `(B*S, vocab)`. Likewise `targets.view(-1)` flattens `(B, S)` → `(B*S,)`. The net effect is to **collapse the batch and sequence axes into a single "one row per token" axis**.

**Why flatten at all.** `F.cross_entropy` expects `(N, C)` logits and `(N,)` targets — `N` independent examples, each scored over `C` classes. Here *every* `(batch, position)` pair is one example: a single next-token prediction. There are `B*S` of them, so we reshape to `N = B*S` rows of `vocab` logits and `B*S` labels; the loss is computed per token and averaged. So a `(2, 4)` batch with `vocab=10` becomes `(8, 10)` logits and `(8,)` targets → 8 next-token predictions scored at once.

**Targets are integer class indices, NOT one-hot.** Each target is just the *index* of the correct next token (e.g. `42` means "the right next token is token #42") — a `LongTensor` of shape `(B*S,)`, not a one-hot vector. Internally `cross_entropy` is `log_softmax` followed by picking out the correct class's log-prob: `−log softmax(logits)[correct]`, averaged over tokens. That's mathematically identical to one-hot cross-entropy (`−Σ onehot·log softmax`) but skips *building* the one-hot and just indexes the right entry — the same "index instead of a wasteful one-hot" trick as `nn.Embedding`. (The `targets` themselves are the *next* token at each position — the input sequence shifted by one.)

*(Aside: modern `F.cross_entropy` can also take float "soft" targets of shape `(N, C)` — for label smoothing or distillation — but the standard, and what we use here, is integer class indices.)*

In [12]:
# pick the fastest available device; all tensors + the model must live on the same one
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

# --- toy corpus: a short string repeated, so there is an obvious pattern to learn quickly ---
text  = "hello world. " * 2000
chars = sorted(set(text))                         # the vocabulary: every distinct character
stoi  = {c: i for i, c in enumerate(chars)}       # char -> integer id (our "tokenizer")
data  = torch.tensor([stoi[c] for c in text], device=device)   # encode the whole text as 1-D ids

def get_batch(bs, block):                          # sample a batch of (input, target) windows
    # bs random start positions; -block-1 leaves room for a full window plus its shifted target
    ix = torch.randint(len(data) - block - 1, (bs,))
    x = torch.stack([data[i:i + block]     for i in ix])   # (bs, block): the input tokens
    y = torch.stack([data[i + 1:i + 1 + block] for i in ix])   # (bs, block): same window shifted by 1
    return x.to(device), y.to(device)              # y[t] is the token that FOLLOWS x[t] — the target

torch.manual_seed(0)                               # reproducible init + batch sampling
model = TinyGPT(vocab=len(chars), block_size=64).to(device)     # build model, move params to device
opt   = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.1)   # optimizer over all params

for step in range(300):
    x, y = get_batch(bs=16, block=64)              # a fresh batch of windows each step
    _, loss = model(x, y)                          # forward: returns (logits, loss); we need the loss
    opt.zero_grad()                                # clear last step's grads (they accumulate)
    loss.backward()                                # backprop: fill every parameter's .grad
    opt.step()                                     # AdamW update using those grads
    if step % 50 == 0:
        print(step, round(loss.item(), 3))         # .item() pulls the scalar loss off the tensor
# loss starts near ln(vocab) (uniform guess) and falls sharply as it learns the repeating pattern

0 2.204
50 0.329
100 0.086
150 0.006
200 0.013
250 0.007


## Running the trained model — inference

Two habits for inference: put the model in **`eval()`** mode and wrap the forward pass in **`torch.no_grad()`** (no autograd graph → faster, less memory). Calling `model(idx)` with **no targets** returns the raw `(B, S, vocab)` logits; the "what comes next" prediction lives at the **last position**, `logits[:, -1]`.

The model handles **variable-length inputs for free**: `forward` builds position ids with `arange(S)`, so any prompt length up to `block_size` works — the sequence axis `S` isn't fixed. Below we feed prompts of different lengths and different characters (all drawn from the tiny corpus's vocabulary).

In [13]:
model.eval()                                   # eval mode (turns off dropout etc.; good practice)
itos = {i: c for c, i in stoi.items()}         # id -> char, to decode the model's outputs
def encode(s): return torch.tensor([[stoi[c] for c in s]], device=device)  # str -> (1, len) ids

@torch.no_grad()                               # no autograd graph during inference
def next_char(prompt):
    logits = model(encode(prompt))             # no targets -> returns (1, S, vocab) logits
    return itos[logits[0, -1].argmax().item()] # greedy: argmax over the LAST position's vocab logits

# prompts of DIFFERENT lengths and DIFFERENT characters (all in-vocab):
for p in ["d", "he", "world", "hello wor", "hello world. hell"]:
    print(f"prompt {p!r:20} (len {len(p):>2}) -> predicted next char {next_char(p)!r}")

prompt 'd'                  (len  1) -> predicted next char '.'
prompt 'he'                 (len  2) -> predicted next char 'l'
prompt 'world'              (len  5) -> predicted next char '.'
prompt 'hello wor'          (len  9) -> predicted next char 'l'
prompt 'hello world. hell'  (len 17) -> predicted next char 'o'


In [14]:
@torch.no_grad()
def generate(prompt, n=40, temperature=1.0):
    model.eval()
    idx = encode(prompt)                                     # (1, S) starting tokens
    for _ in range(n):
        idx_cond = idx[:, -model.block_size:]                # keep only the last block_size tokens (context cap)
        logits = model(idx_cond)                             # (1, S', vocab)
        probs = F.softmax(logits[:, -1] / temperature, dim=-1)   # distribution over the NEXT token
        nxt = torch.multinomial(probs, num_samples=1)            # sample one token id (temperature<1 = greedier)
        idx = torch.cat([idx, nxt], dim=1)                   # append it, then continue from the longer sequence
    return "".join(itos[i] for i in idx[0].tolist())

for p in ["hello", "wor", "d"]:                              # different starting prompts
    print(f"{p!r:9} -> {generate(p, n=30)!r}")

'hello'   -> 'hello world. hello world. hello wor'
'wor'     -> 'world. hello world. hello world. '
'd'       -> 'd. hello world. hello world. he'


> **No KV cache here — deliberately.** This `generate` re-runs the whole model on the entire sequence every step, recomputing all past tokens' keys/values (they never change in a causal model) — simple but `O(N²)` wasteful. Real inference **caches** each layer's past K/V and feeds only the new token per step (a *prefill* pass, then one-token *decode* steps). That cache grows with context and dominates inference memory — exactly what GQA (fewer KV heads) shrinks. Full treatment: Part 9.2.

## What you built — and the two habits that carried it

You assembled a real (if tiny) GPT from scratch and trained it: **scaled dot-product attention → multi-head attention → a pre-norm block → token + position embeddings with a tied head → training loop → generation.** Structurally it's the same model as the nanoGPT in the gradient-checkpointing walkthrough ([../part2_neural_network_fundamentals/2.2_backpropagation/supplementary/04_gradient_checkpointing.ipynb](../part2_neural_network_fundamentals/2.2_backpropagation/supplementary/04_gradient_checkpointing.ipynb)) and the reference architecture in [../ARCHITECTURE.md](../ARCHITECTURE.md) — only smaller.

Two habits did most of the work and carry over to all Transformer code:

- **Check from-scratch pieces against PyTorch's built-ins.** The `sdpa_manual` vs `F.scaled_dot_product_attention` `allclose` was the real test of Step 1 — if they diverge, you have a scaling, transpose, or softmax-axis bug. Do this whenever you reimplement something.
- **Print shapes at every step.** Most Transformer bugs are a stray `transpose`/`view` or a missing `keepdim`; reading shapes is the debugging skill this whole schedule builds toward.

## Extend it

Small changes that deepen the picture:

1. **SwiGLU FFN** — swap the GELU MLP for `W_down( SiLU(x W_gate) ⊙ (x W_up) )` at hidden width `≈ 8D/3`, and confirm it still trains (ties to [../part2_neural_network_fundamentals/2.1_mlp_building_block/02_activations.md](../part2_neural_network_fundamentals/2.1_mlp_building_block/02_activations.md)).
2. **Attention fully from scratch** — replace `F.scaled_dot_product_attention` with your `sdpa_manual`, adding a causal mask (`scores.masked_fill(~tril, float('-inf'))`), and confirm training is unchanged.
3. **Sampling knobs** — `generate` above already takes a `temperature`; try a range (0.5 / 1.0 / 1.5) and add **top-k** sampling, and watch how the samples change.
4. **Scale it up** — more layers, a longer `block_size`, or a bigger corpus; watch the loss curve and the generated text improve.

## Other PyTorch essentials to look into

You now have the full core loop — build → train → evaluate → infer. A handful of building blocks show up in nearly every real project; pointers to look up (deliberately not covered in depth):

- **Save / load:** `torch.save`, `torch.load`, `model.state_dict()` / `load_state_dict()` — persist and restore weights (and optimizer state, to resume training).
- **Training-loop extras:** `torch.nn.utils.clip_grad_norm_` (gradient clipping — Part 2.4/03), `torch.optim.lr_scheduler` (warmup / cosine — Part 2.4/02), **gradient accumulation** (sum grads over several micro-batches before `step()` for a larger effective batch), `nn.Dropout` (Part 2.5).
- **Speed / memory:** `torch.autocast` + `GradScaler` (mixed precision — Part 2.4/05), `torch.compile` (fuses/optimizes the graph), `torch.inference_mode()` (a stricter, faster `no_grad`).
- **Data:** custom `Dataset` / `DataLoader` with a `collate_fn`; **padding** variable-length sequences plus an attention **mask** so padding isn't attended to; `num_workers` for parallel loading.
- **Tensor ops you'll reach for constantly:** `cat` / `stack` / `split` / `chunk`, `gather` / `scatter`, `masked_fill`, `triu` / `tril` (causal masks), `unsqueeze` / `squeeze` / `expand`, `einsum`, `argmax` / `topk`.
- **Debugging:** print shapes; `torch.allclose` to check equivalences; inspect `.grad` / `.grad_fn`; `torch.autograd.set_detect_anomaly(True)` to localize NaNs.
- **At scale (later):** `DistributedDataParallel` / **FSDP** for multi-GPU training — Part 12.

That's the tour. You can now read, run, and modify the curriculum's code — head back to [review_outline.md](../review_outline.md).